<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [161]:
import numpy as np
import torch
from typing import Any

# Travel Problem

### Problem Formulation (Page 5)

In [162]:
class Step:
  """Represents taking an `action`, incurring some `cost` and ending up in a new `state`."""
  def __init__(self, action: Any, cost: float, state: Any):
    self.action = action
    self.cost = cost
    self.state = state

In [163]:
class SearchProblem:
  """Formally and fully represents a search problem."""
  def start_state(self) -> Any:
    raise NotImplementedError
  def successors(self, state: Any) -> list[Step]:
    raise NotImplementedError
  def is_end(self, state: Any) -> bool:
    raise NotImplementedError

In [164]:
class TravelSearchProblem(SearchProblem):
  """An instance of a `SearchProblem` where you try to go from 1 to n in the least time."""
  def __init__(self, num_locs: int):
    self.num_locs = num_locs
  def start_state(self) -> int:
    # Where we start (location 1)
    return 1
  def successors(self, state: int) -> list[Step]:
    """Return possible actions and their costs and resulting states."""
    successors = []
    if state + 1 <= self.num_locs:  # Stay within bounds?
        successors.append(Step(action="walk", cost=1, state=state + 1))
    if 2 * state <= self.num_locs:  # Stay within bounds?
        successors.append(Step(action="tram", cost=2, state=2 * state))
    return successors
  def is_end(self, state: int) -> bool:
    # Have we reached the destination?
    return state == self.num_locs

In [165]:
problem = TravelSearchProblem(num_locs=10)
state = problem.start_state()  # Where we start
successors = problem.successors(state)  # From each state, where can we go
is_end = problem.is_end(successors[0].state)  # Are we done?

# This block을 가지고 놀아보세요. e.g., print(successors[1].cost)

# Exhuastive Search

### Algorithm (Page 8)

In [166]:
class Solution:
  """Represents a solution to a search problem (sequence of actions that produces a cost)."""
  steps: list[Step]
  cost: float
  def __init__(self, steps: list[Step]):
    self.steps = steps
    # The cost of a solution is the sum of the costs of the actions
    costs = [step.cost for step in steps]
    self.cost = sum(costs)

In [167]:
def future_solution(state: Any, num_explored) -> Solution:
  """Return the best solution from `state` (its cost is the future cost)."""
  """Depth First Search Style Exhuastive Search"""
  # Keep track of how many states we've explored
  num_explored += 1
  if problem.is_end(state):
      # Base: already at the end, don't need to take any more actions
      best_solution = Solution(steps=[])
  else:
      # Where can we go?
      successors = problem.successors(state)
      # Flesh each successor out recursively into a solution
      solutions = []
      for first_step in successors:
          future_steps = future_solution(first_step.state, num_explored).steps
          solutions.append(Solution(steps=[first_step] + future_steps))
      # Pick the best one
      best_solution = min(solutions, key=lambda x: x.cost)
  return best_solution

In [168]:
def exhaustive_search(problem: SearchProblem) -> tuple[Solution | None, int]:
  """Perform exhaustive search on `problem` to find the minimum cost solution."""
  # Keep track of how many states we've explored (time complexity)
  num_explored = 0
  # Helper function for the recurrence

  state = problem.start_state()
  solution = future_solution(state,num_explored)
  return solution, num_explored

### Solution (Page 9)

In [169]:
problem = TravelSearchProblem(num_locs=4)
solution, num_explored = exhaustive_search(problem)

In [170]:
print("Total state changes", len(solution.steps))

path_str = "1"
for step in solution.steps:
    path_str += " → " + str(step.state)
print(path_str)
print("Total cost:", solution.cost)

Total state changes 3
1 → 2 → 3 → 4
Total cost: 3


# Dynamic Programming

In [171]:
def future_solution(state: Any, cache:dict[Any, Solution], num_explored) -> Solution:
  """Return the best solution from `state` (its cost is the future cost)."""
  # NEW: check cache first
  if state in cache:
    return cache[state]

  # Keep track of how many states we've explored
  num_explored += 1
  if problem.is_end(state):
    # Base: already at the end, don't need to take any more actions
    best_solution = Solution(steps=[])
  else:
    # Where can we go?
    successors = problem.successors(state)
    # Flesh each successor out recursively into a solution
    solutions = []
    for first_step in successors:
      future_steps = future_solution(first_step.state, cache, num_explored).steps
      solutions.append(Solution(steps=[first_step] + future_steps))
    # Pick the best one
    best_solution = min(solutions, key=lambda x: x.cost)
  # NEW: cache the solution
  cache[state] = best_solution
  return best_solution

In [172]:
def dynamic_programming(problem: SearchProblem) -> tuple[Solution | None, int, dict[Any, Solution]]:
    """Perform dynamic programming on `problem` to find the minimum cost solution."""
    # Keep track of how many states we've explored (time complexity)
    num_explored = 0

    # NEW: cache solutions for each state
    cache: dict[Any, Solution] = {}  # From state -> future solution
    # Helper function for the recurrence

    state = problem.start_state()
    solution = future_solution(state, cache, num_explored)
    return solution, num_explored, cache

### Solution (Page 10)

In [173]:
problem = TravelSearchProblem(num_locs=4)
solution, num_explored, cache = dynamic_programming(problem)

In [174]:
print("Total state changes", len(solution.steps))

path_str = "1"
for step in solution.steps:
    path_str += " → " + str(step.state)
print(path_str)
print("Total cost:", solution.cost)

Total state changes 3
1 → 2 → 3 → 4
Total cost: 3


In [175]:
problem = TravelSearchProblem(num_locs=17)
solution, num_explored, cache = dynamic_programming(problem)

In [176]:
print("Total state changes", len(solution.steps))

path_str = "1"
for step in solution.steps:
    path_str += " → " + str(step.state)
print(path_str)
print("Total cost:", solution.cost)

Total state changes 6
1 → 2 → 3 → 4 → 8 → 16 → 17
Total cost: 8


# Heuristic Search Methods

### Best of N

In [177]:
import random

def uniform_policy(problem: SearchProblem, state: Any) -> Step:
  """Chooses an action uniformly from the successors of `state`."""
  successors = problem.successors(state)
  successor = random.choice(successors)
  return successor

In [178]:
def rollout(problem: SearchProblem, policy, max_steps: int = 10) -> Solution:
    """Sample a policy from the start state of `problem`."""
    state = problem.start_state()
    steps = []
    while not problem.is_end(state) and len(steps) < max_steps:
        # Take a step
        step = policy(problem, state)
        steps.append(step)
        # Advance the state
        state = step.state
    return Solution(steps=steps)

In [179]:
def best_of_n(problem: SearchProblem, policy, num_candidates: int, max_steps: int = 10) -> tuple[Solution | None, int]:
  """
  Perform best-of-n search to `problem`.
  Return the best solution and the number of states explored.
  """
  num_explored = 0
  solutions = []
  for _ in range(num_candidates):
    solution = rollout(problem, policy, max_steps=max_steps)
    solutions.append(solution)
    num_explored += len(solution.steps)
  # For debugging
  final_steps = [solution.steps[-1] for solution in solutions]
  # Choose the best solution
  best_solution = min(solutions, key=lambda x: x.cost)
  return best_solution, num_explored

In [180]:
problem = TravelSearchProblem(num_locs=17)
solution, num_explored = best_of_n(problem, uniform_policy, num_candidates=10)

In [181]:
print("Total state changes", len(solution.steps))

path_str = "1"
for step in solution.steps:
    path_str += " → " + str(step.state)
print(path_str)
print("Total cost:", solution.cost)

Total state changes 8
1 → 2 → 3 → 6 → 7 → 14 → 15 → 16 → 17
Total cost: 11


### Beam search

In [182]:
def beam_search(problem: SearchProblem, beam_width: int, max_steps: int) -> tuple[Solution | None, int]:
  """Perform beam search on `problem` keeping `beam_width` candidates and `max_steps`."""
  # Keep track of how many states we've explored
  num_explored = 0
  candidates = [Solution(steps=[])]
  for step in range(max_steps):
    # Given the existing candidates, expand them by one step
    new_candidates = []
    for candidate in candidates:
      state = candidate.steps[-1].state if candidate.steps else problem.start_state()
      if problem.is_end(state):  # If we've already reached the end, just keep
        new_candidates.append(candidate)
      else:
        # Try all possible actions from `state`
        for successor in problem.successors(state):
          new_candidates.append(Solution(steps=candidate.steps + [successor]))
          num_explored += 1
    # Take the `beam_width` best candidates (lowest cost)
    new_candidates.sort(key=lambda x: x.cost)  # Sort
    candidates = new_candidates[:beam_width]  # Prune
  # Keep only candidates that are done
  candidates = [candidate for candidate in candidates if problem.is_end(candidate.steps[-1].state)]
  return candidates[0], num_explored

In [183]:
problem = TravelSearchProblem(num_locs=17)
solution, num_explored = beam_search(problem, beam_width=2, max_steps=10)

In [184]:
print("Total state changes", len(solution.steps))

path_str = "1"
for step in solution.steps:
    path_str += " → " + str(step.state)
print(path_str)
print("Total cost:", solution.cost)

Total state changes 9
1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 16 → 17
Total cost: 10


# Search in Modern LLM

In [185]:
def is_complete_sentence(state: str) -> bool:
  return state.endswith(")")

In [186]:
def contains_number(state: str) -> bool:
  try:
    eval(state)  # Dangerous!!!
    return True
  except:
    return False

In [187]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F

class LanguageModelSearchProblem(SearchProblem):
  def __init__(self, prompt: str, model_id: str = "Qwen/Qwen3-0.6B"):
    self.prompt = prompt
    self.model_id = model_id
    # Tokenizer converts string to list of integers (and back)
    self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
    # Load the model
    self.model = AutoModelForCausalLM.from_pretrained(self.model_id, dtype=torch.float16).eval()
  def start_state(self) -> str:
    """State starts with prompt."""
    return self.prompt

  def successors(self, state: str) -> list[Step]:
    """Return successors from `state`."""
    # Tokenize the state (prompt + prefix of the response so far)
    input_ids = self.tokenizer(state, return_tensors="pt")["input_ids"]
    # Get probabilities over next token
    all_logits = self.model(input_ids=input_ids).logits  # Logits for all tokens
    next_token_logits = all_logits[0, -1, :]  # Get the logits for the last token
    next_token_probs = F.softmax(next_token_logits, dim=-1)  # Convert to probabilities
    topk = torch.topk(input=next_token_probs, k=5)  # Keep only the top 5 tokens
    # Build a successor for each of the top 5 tokens
    successors = []
    for index, prob in zip(topk.indices, topk.values):
      action = index.item()
      # Maximize product of probabilities = minimize sum of log probabiltiies (costs)
      cost = -torch.log(prob).item()
      # Here's where we end up
      new_state = state + self.tokenizer.decode([action])
      # If the resulting state is the end and has a number, get a big reward (negative cost)
      if is_complete_sentence(new_state) and contains_number(new_state):
          cost -= 100
      successors.append(Step(action=action, cost=cost, state=new_state))
    return successors
  def is_end(self, state: str) -> bool:
    """We're done once we get well-formed JSON."""
    return is_complete_sentence(state)

In [188]:
def lm_policy(problem: LanguageModelSearchProblem, state: str) -> Step:
  """Sample the next token given the tokens so far (`state`)."""
  # Get the successors from the state
  successors = problem.successors(state)
  # Get the costs for all the successors
  costs = [successor.cost for successor in successors]
  # Convert costs to probabilities
  probs = torch.softmax(-torch.tensor(costs), dim=-1)

  # Sample an element from the `probs` distribution
  index = torch.multinomial(probs, num_samples=1)[0]
  # Return the corresponding successor
  successor = successors[index]
  return successor

In [189]:
problem = LanguageModelSearchProblem( prompt="(3 + 7 *" )
step = lm_policy(problem, problem.start_state())
torch.manual_seed(1)
solution, num_explored = best_of_n(problem, lm_policy, num_candidates=5, max_steps=10)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [190]:
print("Total state changes", len(solution.steps))

path_str = "1"
for step in solution.steps:
    path_str += " → " + str(step.state)
print(path_str)
print("Total cost:", solution.cost)

Total state changes 3
1 → (3 + 7 *  → (3 + 7 * 2 → (3 + 7 * 2)
Total cost: -97.683349609375
